# VisDrone — yolo26**n**

Lam tron mot size: baseline -> prune 50% -> finetune + CWD -> val.
Ket qua la **hai dong** cua bang: `YOLO26-N` va `Ours-N`.

| | |
|---|---|
| Baseline | `yolo26n.pt` (COCO), 100 epoch tren VisDrone |
| Ours | L1-norm uniform 50% (div 8) + 100 epoch CWD tau=9, kd_layers=neck |
| Batch / imgsz / seed | 16 / 640 / 0 |
| cos_lr / patience / warmup | False / 100 / 3.0 |
| Uoc tinh | ~9h, **1-2 phien** |

> Bon notebook n/s/m/l dung **y het** cac tham so nay. Doi mot cai thoi la ca
> bang het so sanh duoc.

## Cach chay

1. Settings -> Accelerator **GPU T4 x2**, **Internet: On**
2. **Save & Run All**. Lan dau khong can Add Data.
3. Phien tu dung o 10h. Cell cuoi bao con thieu bao nhieu epoch -> Add Data
   output cua chinh lan chay nay roi Save & Run All lai.
4. Xong thi gui lai bang 2 dong o cell cuoi.

Logic nam trong `notebooks/share_visdrone/vd_common.py` trong repo, cell setup
tu `git pull` moi lan chay — sua loi o do la ca nhom co ngay, khong phai import
lai notebook.

Repo: https://github.com/xauskeleton/yolo26_prune_cwd

## 1. Setup

In [ ]:
import os, sys, pathlib, subprocess

REPO_DIR = pathlib.Path("/kaggle/working/yolo")
# Luon keo ban moi nhat: logic nam trong repo nen sua o do la ca nhom co ngay.
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/xauskeleton/yolo26_prune_cwd", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

# Phai dung fork nay, KHONG "pip install ultralytics": checkpoint sau khi prune
# duoc pickle voi ultralytics.nn.tasks_pruned nen ban chinh thuc khong load duoc.
for d in (REPO_DIR, REPO_DIR / "pruning", REPO_DIR / "notebooks" / "share_visdrone"):
    sys.path.insert(0, str(d))
# DDP sinh tien trinh con chay file tam ngoai repo -> phai truyen qua PYTHONPATH.
os.environ["PYTHONPATH"] = str(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", "requirements.txt"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".",
                "--no-deps"], check=False)

import torch
import vd_common as V
print("GPU:", torch.cuda.device_count(), "| vd_common:", V.__file__)

## 2. Cau hinh

In [ ]:
SIZE   = "n"
MODEL  = "yolo26n.pt"
RATIO  = 0.5

BASE_NAME = "vd_yolo26n"
OURS_NAME = "vd_oursn"
PRUNED = REPO_DIR / "weights" / "yolo26n_vd_pruned50.pt"

V.init(
    REPO_DIR = REPO_DIR,
    DATA     = "VisDrone.yaml",   # Ultralytics tu tai 2.3 GB, can Internet: On
    EPOCHS   = 100,
    BATCH    = 16,
    IMGSZ    = 640,
    DEVICE   = "0,1" if torch.cuda.device_count() > 1 else "0",
    # Chot cung thay vi de mac dinh: cac run VOC truoc day khong dong nhat
    # (n/s/l dung batch 32 + cos_lr=True + patience 20-30, m dung 16/False/100).
    COS_LR   = False,
    PATIENCE = 100,
    WARMUP   = 3.0,
    # Kaggle giet phien o 12h va phien bi giet thi KHONG luu output.
    STOP_AFTER_H = 10.0,
)

# Chi dung khi chi co moi mot file last.pt roi le (phien bi danh dau failed).
# Upload ca thu muc <run_name>/weights/ thi khong can dien gi.
MANUAL_LAST = {BASE_NAME: "", OURS_NAME: ""}

print(BASE_NAME, "|", OURS_NAME, "|", V.CFG["DEVICE"])

## 3. Resume

In [ ]:
V.restore(BASE_NAME, OURS_NAME, PRUNED, MANUAL_LAST)

## 4. Baseline yolo26n

In [ ]:
n_base = V.train(BASE_NAME, MODEL)

if n_base < 100:
    print()
    print("Baseline moi {}/100 epoch - het gio phien nay.".format(n_base))
    print("Add Data output lan nay roi Save & Run All lai. Cac cell duoi bo qua.")

## 5. Prune 50%

In [ ]:
BEST_BASE = REPO_DIR / "runs" / BASE_NAME / "weights" / "best.pt"

if n_base < 100:
    print("Bo qua: baseline chua xong.")
else:
    V.prune50(SIZE, BEST_BASE, PRUNED, RATIO)

## 6. Finetune + CWD

In [ ]:
if n_base < 100 or not PRUNED.exists():
    print("Bo qua: chua co model da prune.")
    n_ours = 0
else:
    # Teacher la baseline cua chinh size nay.
    n_ours = V.train(OURS_NAME, str(PRUNED),
                     finetune=True,      # build DetectionModelPruned tu maskbndict
                     kd=True, kd_teacher=str(BEST_BASE), kd_method="cwd",
                     kd_lambda=0.5, kd_layers="neck", kd_warmup=5,
                     cwd_temperature=9.0)

## 7. Ket qua

In [ ]:
V.report(SIZE, BASE_NAME, OURS_NAME)